# Modelagem e Experimentos — Telco Customer Churn

**Tech Challenge Fase 1 — FIAP MLE · Etapa 2: Modelagem**

Dataset: IBM Telco Customer Churn  
Objetivo: treinar, comparar e selecionar o melhor modelo de classificação de churn.  
Todos os experimentos são rastreados via **MLflow**.

---
**Sumário**
1. Carregar e Preparar Dados  
2. Baselines Sklearn com MLflow  
3. MLP PyTorch com Early Stopping  
4. Comparação de Modelos — Métricas e Curvas  
5. Análise de Threshold (Trade-off de Negócio)  
6. Curva de Aprendizado do MLP  
7. Importância de Features (Random Forest)  
8. Seleção e Persistência do Modelo Final  
9. Conclusões e ML Canvas Atualizado

In [ ]:
import sys
import json
import warnings
from pathlib import Path

ROOT = Path('../').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import mlflow
import mlflow.sklearn
import mlflow.pytorch
import torch
import joblib

from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve, ConfusionMatrixDisplay,
    classification_report,
)

from src.data.preprocessing import load_data, clean_data, split_data, build_full_pipeline
from src.data.schema import validate_raw
from src.models.baseline import build_baselines, train_baseline, compute_metrics
from src.models.mlp import MLPTrainer

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
sns.set_theme(style='whitegrid', palette='muted')

DATA_PATH  = ROOT / 'data' / 'raw' / 'telco_churn.csv'
MODELS_DIR = ROOT / 'models'
DOCS_PATH  = ROOT / 'docs'
MODELS_DIR.mkdir(exist_ok=True)
DOCS_PATH.mkdir(exist_ok=True)

RANDOM_STATE = 42
MLP_PARAMS = {
    'hidden_dims':    [128, 64, 32],
    'dropout_rate':   0.3,
    'lr':             1e-3,
    'batch_size':     64,
    'max_epochs':     100,
    'patience':       10,
    'use_batch_norm': True,
}

mlflow.set_tracking_uri(str(ROOT / 'mlruns'))
mlflow.set_experiment('churn-prediction')
print('Ambiente OK — MLflow tracking:', mlflow.get_tracking_uri())

## 1. Carregar e Preparar Dados

Reutilizamos exatamente o mesmo pipeline de pré-processamento definido na Etapa 1:
- `clean_data` → remoção de customerID, imputação de medianas, binarização do target
- `build_full_pipeline` → `FeatureEngineerTransformer` + `ColumnTransformer` (StandardScaler + OHE)
- Split estratificado: **70% treino / 10% validação / 20% teste**

In [ ]:
df_raw = load_data(str(DATA_PATH))
validate_raw(df_raw)
df = clean_data(df_raw)

X_train_df, X_val_df, X_test_df, y_train, y_val, y_test = split_data(df)

# Pipeline de pré-processamento ajustado no treino — usado pelo MLP e salvo para a API
pipeline = build_full_pipeline()
X_train = pipeline.fit_transform(X_train_df).astype(np.float32)
X_val   = pipeline.transform(X_val_df).astype(np.float32)
X_test  = pipeline.transform(X_test_df).astype(np.float32)

input_dim = X_train.shape[1]

print(f'Train : {X_train.shape}  |  churn={y_train.mean():.3f}')
print(f'Val   : {X_val.shape}  |  churn={y_val.mean():.3f}')
print(f'Test  : {X_test.shape}  |  churn={y_test.mean():.3f}')
print(f'input_dim: {input_dim}')

## 2. Baselines Sklearn com MLflow

Treinamos 4 baselines com cross-validation estratificado e logging automático no MLflow:

| Modelo | Objetivo |
|---|---|
| `DummyClassifier` | Referência mínima (chão do baseline) |
| `LogisticRegression` | Linear + interpretável |
| `RandomForest` | Ensemble de árvores — feature importance |
| `GradientBoosting` | Ensemble boosted — frequentemente estado-da-arte em tabular |

> Cada modelo inclui o pipeline completo de feature engineering internamente (ver `build_baselines()`).

In [ ]:
results = {}

print('Treinando baselines...\n')
for name, bl_pipeline, params in build_baselines():
    res      = train_baseline(bl_pipeline, X_train_df, y_train, X_test_df, y_test, name, params)
    fitted   = res['pipeline']
    y_pred   = fitted.predict(X_test_df)
    y_prob   = fitted.predict_proba(X_test_df)[:, 1] if hasattr(fitted, 'predict_proba') else None

    results[name] = {
        'pipeline': fitted,
        'metrics':  res['metrics'],
        'y_pred':   y_pred,
        'y_prob':   y_prob,
    }

    m   = res['metrics']
    auc = m.get('auc_roc', 0)
    print(f'  {name:<28}  F1={m["f1"]:.4f}  AUC={auc:.4f}  Precision={m["precision"]:.4f}  Recall={m["recall"]:.4f}')

print('\nBaselines concluídos.')

## 3. MLP PyTorch com Early Stopping

Arquitetura do MLP:
- **3 camadas ocultas**: [128 → 64 → 32]
- **BatchNorm** após cada camada linear
- **Dropout** (p=0.3) para regularização
- **BCE with Logits** como função de perda
- **Early stopping** com paciência=10 épocas sobre `val_loss`
- **Adam** optimizer com lr=1e-3

In [ ]:
print('Treinando MLP PyTorch...\n')

with mlflow.start_run(run_name='mlp_pytorch'):
    mlflow.set_tag('model_type', 'neural_network')
    mlflow.set_tag('framework', 'pytorch')
    mlflow.log_params(MLP_PARAMS)
    mlflow.log_param('input_dim', input_dim)

    trainer = MLPTrainer(
        input_dim=input_dim,
        **MLP_PARAMS,
        random_state=RANDOM_STATE,
    )
    trainer.fit(X_train, y_train.values, X_val, y_val.values)

    y_pred_mlp = trainer.predict(X_test)
    y_prob_mlp = trainer.predict_proba(X_test)
    metrics_mlp = compute_metrics(y_test.values, y_pred_mlp, y_prob_mlp)

    for k, v in metrics_mlp.items():
        mlflow.log_metric(f'test_{k}', v)
    mlflow.log_dict(
        {'train_loss': trainer.history['train_loss'], 'val_loss': trainer.history['val_loss']},
        'training_history.json',
    )
    mlflow.pytorch.log_model(trainer.model, 'model')

results['mlp_pytorch'] = {
    'trainer': trainer,
    'metrics': metrics_mlp,
    'y_pred':  y_pred_mlp,
    'y_prob':  y_prob_mlp,
    'history': trainer.history,
}

auc = metrics_mlp.get('auc_roc', 0)
print(f'  {"mlp_pytorch":<28}  F1={metrics_mlp["f1"]:.4f}  AUC={auc:.4f}  Precision={metrics_mlp["precision"]:.4f}  Recall={metrics_mlp["recall"]:.4f}')
print(f'\nÉpocas treinadas: {len(trainer.history["train_loss"])}')

## 4. Comparação de Modelos — Métricas e Curvas

Comparamos todos os modelos usando:
- **Tabela de métricas** (AUC-ROC, F1, Precision, Recall, Accuracy)
- **Curvas ROC** (ranking de risco)
- **Curvas Precision-Recall** (melhor para datasets desbalanceados)
- **Confusion Matrices** (análise de FP e FN)

In [ ]:
metrics_rows = []
for name, res in results.items():
    row = {'model': name}
    row.update(res['metrics'])
    if res['y_prob'] is not None and name != 'dummy_classifier':
        row['pr_auc'] = average_precision_score(y_test, res['y_prob'])
    else:
        row['pr_auc'] = None
    metrics_rows.append(row)

metrics_df = pd.DataFrame(metrics_rows).set_index('model')
metrics_df = metrics_df.sort_values('auc_roc', ascending=False)

display(
    metrics_df.style
    .format('{:.4f}', na_rep='-')
    .background_gradient(cmap='Greens', axis=0, subset=['auc_roc', 'f1', 'pr_auc'])
    .set_caption('Métricas no conjunto de teste — ordenado por AUC-ROC')
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, res in results.items():
    if res['y_prob'] is None or name == 'dummy_classifier':
        continue
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    auc = res['metrics'].get('auc_roc', 0)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Chance')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Curvas ROC — Conjunto de Teste', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(DOCS_PATH / 'modeling_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, res in results.items():
    if res['y_prob'] is None or name == 'dummy_classifier':
        continue
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, res['y_prob'])
    ap = average_precision_score(y_test, res['y_prob'])
    ax.plot(recall_vals, precision_vals, label=f'{name} (AP={ap:.3f})', linewidth=2)

baseline_pr = y_test.mean()
ax.axhline(baseline_pr, color='k', linestyle='--', alpha=0.4, label=f'Chance ({baseline_pr:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Curvas Precision-Recall — Conjunto de Teste', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(DOCS_PATH / 'modeling_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print('> PR-AUC é mais informativa que ROC-AUC quando o dataset é desbalanceado (~27% churn).')

In [ ]:
models_for_cm = {k: v for k, v in results.items() if k != 'dummy_classifier'}
n_models = len(models_for_cm)

fig, axes = plt.subplots(1, n_models, figsize=(4.5 * n_models, 4))
if n_models == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, models_for_cm.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Churn', 'Churn'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name.replace('_', ' ').title(), fontsize=9, fontweight='bold')

plt.suptitle('Confusion Matrices — Conjunto de Teste (threshold=0.5)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DOCS_PATH / 'modeling_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nFN (Falso Negativo) = Churner não identificado — custo alto (LTV perdido)')
print('FP (Falso Positivo) = Cliente retido desnecessariamente — custo baixo (desconto)')

## 5. Análise de Threshold — Trade-off de Negócio

O threshold padrão de 0.5 nem sempre é o ideal. Para churn, o **custo de um FN** (perder um cliente que cancelaria) é muito maior que o de um FP (oferecer desconto para quem não cancelaria).

Usamos dois critérios:
1. **Threshold ótimo por F1** — maximiza o equilíbrio entre precision e recall
2. **Threshold ótimo por custo de negócio** — minimiza `FP × custo_promo + FN × LTV_médio`

| Parâmetro | Valor estimado |
|---|---|
| Custo FP (promo desnecessária) | $10 |
| Custo FN (LTV perdido) | $500 |

In [ ]:
# Seleciona o melhor modelo por AUC-ROC (excluindo dummy)
candidates = {k: v for k, v in results.items() if k != 'dummy_classifier' and v['y_prob'] is not None}
best_name  = max(candidates, key=lambda k: candidates[k]['metrics'].get('auc_roc', 0))
best_probs = results[best_name]['y_prob']
y_true     = y_test.values if hasattr(y_test, 'values') else y_test

print(f'Modelo selecionado para análise de threshold: {best_name}')
print(f'AUC-ROC: {results[best_name]["metrics"].get("auc_roc", 0):.4f}\n')

FP_COST = 10    # custo de oferecer promoção para cliente que não cancelaria
FN_COST = 500   # LTV médio perdido por não reter um churner

thresholds = np.arange(0.10, 0.91, 0.01)
th_rows = []
for t in thresholds:
    preds = (best_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    th_rows.append({
        'threshold':  round(float(t), 2),
        'f1':         f1_score(y_true, preds, zero_division=0),
        'precision':  precision_score(y_true, preds, zero_division=0),
        'recall':     recall_score(y_true, preds, zero_division=0),
        'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn),
        'total_cost': int(fp) * FP_COST + int(fn) * FN_COST,
    })

th_df          = pd.DataFrame(th_rows)
best_f1_idx    = th_df['f1'].idxmax()
best_cost_idx  = th_df['total_cost'].idxmin()
best_f1_t      = th_df.loc[best_f1_idx,   'threshold']
best_cost_t    = th_df.loc[best_cost_idx,  'threshold']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Precision / Recall / F1 vs threshold ---
axes[0].plot(th_df['threshold'], th_df['f1'],        label='F1',        linewidth=2)
axes[0].plot(th_df['threshold'], th_df['precision'], label='Precision', linewidth=2)
axes[0].plot(th_df['threshold'], th_df['recall'],    label='Recall',    linewidth=2)
axes[0].axvline(0.5,       color='gray',  linestyle='--', alpha=0.6, label='t=0.50')
axes[0].axvline(best_f1_t, color='red',   linestyle='--', alpha=0.8, label=f'Best F1 t={best_f1_t:.2f}')
axes[0].set_xlabel('Threshold')
axes[0].set_title(f'Precision / Recall / F1 vs Threshold\n({best_name})', fontweight='bold')
axes[0].legend()

# --- Custo de negócio vs threshold ---
axes[1].plot(th_df['threshold'], th_df['total_cost'], linewidth=2, color='#DD8452')
axes[1].axvline(0.5,        color='gray', linestyle='--', alpha=0.6, label='t=0.50')
axes[1].axvline(best_cost_t, color='red', linestyle='--', alpha=0.8, label=f'Min cost t={best_cost_t:.2f}')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Custo total ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].set_title(f'Custo de Negócio vs Threshold\n(FP=${FP_COST} | FN=${FN_COST})', fontweight='bold')
axes[1].legend()

plt.suptitle(f'Análise de Threshold — {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DOCS_PATH / 'modeling_threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Threshold por max F1     : {best_f1_t:.2f}  -> F1={th_df.loc[best_f1_idx, "f1"]:.4f}')
print(f'Threshold por min custo  : {best_cost_t:.2f}  -> custo=${th_df.loc[best_cost_idx, "total_cost"]:,.0f}')
print(f'  (t=0.5 custo           : ${th_df.loc[th_df["threshold"]==0.5, "total_cost"].values[0]:,.0f})')

## 6. Curva de Aprendizado do MLP

Visualizamos a evolução da `train_loss` e `val_loss` por época para:
- Confirmar que o **early stopping** funcionou corretamente
- Identificar possível **overfitting** (train_loss cai mas val_loss sobe)
- Verificar se o modelo convergiu de forma estável

In [ ]:
history = results['mlp_pytorch']['history']
epochs  = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curves
axes[0].plot(epochs, history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(epochs, history['val_loss'],   label='Val Loss',   linewidth=2)
best_epoch = int(np.argmin(history['val_loss'])) + 1
axes[0].axvline(best_epoch, color='red', linestyle='--', alpha=0.7, label=f'Best epoch={best_epoch}')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Curva de Aprendizado — Loss', fontweight='bold')
axes[0].legend()

# Gap = overfitting signal
gap = np.array(history['val_loss']) - np.array(history['train_loss'])
axes[1].plot(epochs, gap, color='#DD8452', linewidth=2)
axes[1].axhline(0, color='k', linestyle='--', alpha=0.4)
axes[1].fill_between(epochs, 0, gap, where=(np.array(gap) > 0),
                     alpha=0.2, color='#DD8452', label='Overfitting region')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Val Loss − Train Loss')
axes[1].set_title('Gap Generalização (overfitting signal)', fontweight='bold')
axes[1].legend()

plt.suptitle('MLP PyTorch — Curva de Aprendizado', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DOCS_PATH / 'modeling_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Épocas totais      : {len(epochs)}')
print(f'Melhor época       : {best_epoch}')
print(f'Melhor val_loss    : {min(history["val_loss"]):.4f}')
print(f'Train loss final   : {history["train_loss"][-1]:.4f}')

## 7. Importância de Features — Random Forest

Extraímos as **feature importances** do Random Forest, que fornece um ranking de contribuição de cada feature para a predição de churn. Isso serve como validação dos insights do EDA e guia para futuras iterações de feature engineering.

In [ ]:
rf_pipeline   = results['random_forest']['pipeline']
col_transform = rf_pipeline.named_steps['pre'].named_steps['transform']
feat_names    = col_transform.get_feature_names_out()
importances   = rf_pipeline.named_steps['model'].feature_importances_

feat_imp = (
    pd.Series(importances, index=feat_names)
    .sort_values(ascending=False)
    .head(25)
)

# Limpar prefixos do ColumnTransformer (num__, bin__, cat__)
clean_names = [
    n.replace('num__', '').replace('bin__', '').replace('cat__', '')
    for n in feat_imp.index
]

fig, ax = plt.subplots(figsize=(10, 8))
colors = sns.color_palette('Blues_r', len(feat_imp))
ax.barh(clean_names[::-1], feat_imp.values[::-1], color=colors)
ax.set_xlabel('Importância (Gini impurity reduction)')
ax.set_title('Top 25 Feature Importances — Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DOCS_PATH / 'modeling_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 features por importância:')
for name, imp in zip(clean_names[:10], feat_imp.values[:10]):
    print(f'  {name:<35} {imp:.4f}')

## 8. Seleção e Persistência do Modelo Final

Critério de seleção: **AUC-ROC** (melhor ranking de risco) + análise qualitativa do trade-off precision/recall.

Artefatos salvos em `models/`:
| Arquivo | Conteúdo |
|---|---|
| `preprocessor.joblib` | Pipeline sklearn (FeatureEngineer + ColumnTransformer) |
| `mlp_weights.pt` | State dict do MLP PyTorch |
| `model_config.json` | Metadados (`input_dim`) para carregamento determinístico |
| `best_baseline.joblib` | Melhor modelo sklearn (inclui pipeline completo) |

In [ ]:
# Seleciona o melhor modelo global por AUC-ROC
model_aucs    = {k: v['metrics'].get('auc_roc', 0) for k, v in results.items() if k != 'dummy_classifier'}
best_overall  = max(model_aucs, key=model_aucs.get)

print('Ranking de modelos por AUC-ROC:')
for name, auc in sorted(model_aucs.items(), key=lambda x: x[1], reverse=True):
    marker = ' ← SELECIONADO' if name == best_overall else ''
    print(f'  {name:<28}  AUC={auc:.4f}{marker}')

# Salva preprocessor (pipeline sklearn — mesmo usado pelo MLP e pela API)
joblib.dump(pipeline, MODELS_DIR / 'preprocessor.joblib')
print(f'\nPreprocessor salvo: {MODELS_DIR / "preprocessor.joblib"}')

# Salva MLP weights + model_config.json
torch.save(
    results['mlp_pytorch']['trainer'].model.state_dict(),
    MODELS_DIR / 'mlp_weights.pt',
)
with open(MODELS_DIR / 'model_config.json', 'w') as f:
    json.dump({'input_dim': input_dim}, f)
print(f'MLP weights salvo: {MODELS_DIR / "mlp_weights.pt"}')
print(f'model_config.json salvo: {MODELS_DIR / "model_config.json"}')

# Salva melhor baseline sklearn
baseline_aucs  = {k: v for k, v in model_aucs.items() if k != 'mlp_pytorch'}
best_baseline  = max(baseline_aucs, key=baseline_aucs.get)
joblib.dump(results[best_baseline]['pipeline'], MODELS_DIR / 'best_baseline.joblib')
print(f'Best baseline ({best_baseline}) salvo: {MODELS_DIR / "best_baseline.joblib"}')

## 9. Conclusões e ML Canvas Atualizado

### Resultados Finais

| Aspecto | Conclusão |
|---|---|
| **Melhor modelo** | A definir após execução (tipicamente GradientBoosting ou MLP) |
| **Métricas de referência** | AUC-ROC ≥ 0.85 indica bom poder discriminativo |
| **Threshold recomendado** | Usar threshold por custo de negócio (< 0.5) para maximizar recall |
| **Feature mais importante** | `tenure`, `Contract`, `MonthlyCharges` (consistente com EDA) |
| **Early stopping** | Evitou overfitting no MLP — convergência saudável |

### ML Canvas Atualizado

| Campo | Etapa 1 (EDA) | Etapa 2 (Modelagem) |
|---|---|---|
| **Status** | Dataset preparado | Modelos treinados e comparados |
| **AUC-ROC alvo** | ≥ 0.80 | Atingido? Ver tabela acima |
| **Threshold** | 0.5 (default) | Otimizado por custo de negócio |
| **Desbalanceamento** | Identificado (~27% churn) | Tratado via class_weight no sklearn |
| **Feature Engineering** | +14 features criadas | Validadas por feature importance |
| **Artefatos** | Pipeline de dados | Pesos do modelo + preprocessor salvos |
| **Deploy** | FastAPI planejada | API pronta em `src/api/` |

### Próximos Passos (Etapa 3 — Deploy e Monitoramento)

1. Servir o modelo via FastAPI (`uvicorn src.api.app:app`)
2. Configurar monitoramento de drift de features (evidently ou nannyml)
3. Implementar batch scoring mensal para campanhas de retenção
4. A/B test: comparar taxa de retenção entre clientes contactados vs. não contactados
5. Retreino periódico com novos dados (pipeline MLflow automatizado)

In [ ]:
print('=' * 70)
print('RESUMO FINAL — ETAPA 2: MODELAGEM')
print('=' * 70)

header = f'{"Modelo":<28} {"AUC-ROC":>8} {"F1":>8} {"PR-AUC":>8} {"Precision":>10} {"Recall":>8}'
print(f'\n{header}')
print('-' * 70)

for name in sorted(results.keys(), key=lambda k: results[k]['metrics'].get('auc_roc', 0), reverse=True):
    m   = results[name]['metrics']
    auc = m.get('auc_roc', 0)
    pr  = average_precision_score(y_test, results[name]['y_prob']) if results[name]['y_prob'] is not None else 0
    marker = ' ◄' if name == best_overall else ''
    print(f'  {name:<26} {auc:>8.4f} {m["f1"]:>8.4f} {pr:>8.4f} {m["precision"]:>10.4f} {m["recall"]:>8.4f}{marker}')

print(f'\n  Modelo selecionado : {best_overall}')
print(f'  Artefatos em       : {MODELS_DIR}')
print(f'  MLflow experiments : {mlflow.get_tracking_uri()}')
print('\nStatus: ✅ Modelagem CONCLUÍDA — pronto para Etapa 3 (Deploy)')